In [ ]:
import os, subprocess, glob
wh = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if any(x.endswith('.whl') for x in files):
        wh = root
        break
assert wh, 'pip-wheels dataset not found'
r = os.system('pip -q install --no-index --find-links=' + wh + ' anyascii sparse_dot_topn rapidfuzz indic_transliteration')
assert r == 0, 'offline pip install failed'
data = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'train_source1.tsv' in files and root.endswith('/train'):
        data = root[:-6]
        break
assert data, 'amc-2026 dataset not attached'
code = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'run_big.py' in files and root.endswith('/src'):
        code = root[:-4]
        break
assert code, 'er-v5-code dataset not attached'
os.system('cp -r ' + code + '/. /kaggle/working/er/')
os.chdir('/kaggle/working/er')
print('data:', data, 'code:', code, flush=True)
r = os.system('python -u -m src.prep --data ' + data + ' --out /kaggle/working/prep --workers 4')
assert r == 0, 'prep failed'
cfg = '{"train_per_country": 120000, "feat_workers": 4, "seeds": [0, 1]}'
r = os.system("python -X faulthandler -u -m src.run_big --prep /kaggle/working/prep --data " + data + " --out /kaggle/working/out --phase all --cfg '" + cfg + "'")
assert r == 0, 'pipeline failed'
os.system('cp /kaggle/working/out/output/matching_results.tsv /kaggle/working/matching_results_v6.tsv; cp /kaggle/working/out/run_log_train.json /kaggle/working/ 2>/dev/null; cp /kaggle/working/out/run_log_final.json /kaggle/working/ 2>/dev/null')
os.system('mkdir -p /kaggle/working/artifacts && cp /kaggle/working/out/test_*.parquet /kaggle/working/out/oof_pA.npy /kaggle/working/out/loco_pA.npy /kaggle/working/artifacts/ 2>/dev/null; cp -r /kaggle/working/out/models /kaggle/working/artifacts/ 2>/dev/null; cp /kaggle/working/out/*_stats.json /kaggle/working/out/train_prune.json /kaggle/working/artifacts/ 2>/dev/null; ls -la /kaggle/working/artifacts')
os.system('cd /kaggle/working/out/output && zip -q /kaggle/working/candidate_pairs_v6.zip candidate_pairs.tsv')
os.system('rm -rf /kaggle/working/prep /kaggle/working/out /kaggle/working/er')
print('V6 DONE', flush=True)
